In [0]:
import pyspark.sql.functions as f

%md
| Column | Description | Values / Meaning |
|---|---|---|
| `oph` | Operating hours of the engine | — |
| `pist_m` | Piston material | — |
| `issue_type` | Combustion issue type | `1` = typical issue<br>`2` = atypical issue<br>`3` = non-related<br>`4` = non-symptomatic |
| `bmep` | Brake mean effective pressure (average pressure forcing the pistons down) | — |
| `ng_imp` | Natural gas impurities (nmol) | — |
| `past_dmg` | Engine had past damages | `1` = true<br>`0` = false |
| `resting_analysis_results` | Resting results after operation | `0` = normal<br>`1` = abnormal<br>`2` = critical |
| `rpm_max` | Maximum rotations per minute achieved | — |
| `full_load_issues` | Issues induced due to full load operation | `1` = yes<br>`0` = no |
| `number_up` | Number of unplanned events | — |
| `number_tc` | Number of installed turbochargers | — |
| `op_set_1` | Operational engine setting 1 | — |
| `op_set_2` | Operational engine setting 2 | — |
| `op_set_3` | Operational engine setting 3 | — |
| `breakdown` | Breakdown probability indicator | `0` = less chance of breakdown<br>`1` = more chance of breakdown |

In [0]:
import pyspark.sql.types as t


schema = t.StructType(
    [
        t.StructField("engine_id", t.StringType(), True),
        t.StructField("timestamp", t.DateType(), True),
        t.StructField("oph", t.LongType(), True),
        t.StructField("pist_m", t.LongType(), True),
        t.StructField("issue_type", t.LongType(), True),
        t.StructField("bmep", t.DoubleType(), True),
        t.StructField("ng_imp", t.LongType(), True),
        t.StructField("past_dmg", t.BooleanType(), True),
        t.StructField("resting_analysis_results", t.LongType(), True),
        t.StructField("rpm_max", t.LongType(), True),
        t.StructField("full_load_issues", t.BooleanType(), True),
        t.StructField("number_up", t.DoubleType(), True),
        t.StructField("number_tc", t.StringType(), True),
        t.StructField("op_set_1", t.DoubleType(), True),
        t.StructField("op_set_2", t.DoubleType(), True),
        t.StructField("op_set_3", t.DoubleType(), True),
        t.StructField("breakdown", t.BooleanType(), True),
    ]
)

df = spark.read.table("workspace.case_study_test.01_input_data")

# Cast all columns at once to avoid deeply nested execution plans
casts = {field.name: f.col(field.name).cast(field.dataType) for field in schema.fields}
df = df.withColumns(casts)

df = df.filter(f.col("breakdown").isNotNull())
df = df.withColumn("number_tc", f.when(f.col("number_tc")=="1.0", f.lit("single")).when(f.col("number_tc")=="2.0", f.lit("twin")).otherwise(f.col("number_tc")))
issue_type_map = {
    1: "typical issue",
    2: "atypical issue",
    3: "non-related",
    4: "non-symptomatic",
}

df = df.withColumn(
    "issue_type_desc",
    f.expr(
        "CASE issue_type "
        "WHEN 1 THEN 'typical issue' "
        "WHEN 2 THEN 'atypical issue' "
        "WHEN 3 THEN 'non-related' "
        "WHEN 4 THEN 'non-symptomatic' "
        "ELSE NULL END"
    ),
).withColumn("year_month", f.date_format(f.col("timestamp"), "yyyy-MM"))

df.display()

In [0]:
df_engines = df.groupBy("engine_id").agg(
    f.collect_list(
        f.struct(
            f.col("timestamp"),
            f.col("oph"),
            f.col("op_set_1"),
            f.col("op_set_2"),
            f.col("op_set_3"),
            f.col("breakdown"),
            f.col("issue_type_desc"),
            f.col("number_tc")
        )
    ).alias("oph_by_timestamp")
).withColumn(
    "oph_sorted",
    f.expr("array_sort(oph_by_timestamp)")
).select("engine_id", "oph_sorted")


In [0]:
import plotly.graph_objs as go

def plot_engine_issues_plotly(df, tc):
    df = df.select("engine_id", f.expr(f"filter(oph_sorted, x -> x.number_tc = '{tc}')").alias("oph_sorted"))
    pdf = df.select("engine_id", "oph_sorted").toPandas()
    fig = go.Figure()

    for idx, row in pdf.iterrows():
        engine_id = row["engine_id"]
        oph_data = row["oph_sorted"]
        timestamps = [x["timestamp"] for x in oph_data]
        ophs = [x["oph"] for x in oph_data]
        labels = [
            f"issue: {x['issue_type_desc']}, number_tc: {x['number_tc']}, oph: {x['oph']}, op_set_1: {x['op_set_1']}, op_set_2: {x['op_set_2']}, op_set_3: {x['op_set_3']}, breakdown: {x['breakdown']}"
            for x in oph_data
        ]

        fig.add_trace(
            go.Scatter(
                x=timestamps,
                y=ophs,
                mode="lines+markers",
                name=f"Engine {engine_id}",
                hovertext=labels,
                hoverinfo="text",
                marker=dict(size=8),
            )
        )

    fig.update_layout(
        title=f"Engine Issues Over Time and oph tc={tc}",
        xaxis_title="Timestamp",
        yaxis_title="Operating Hours (oph)",
        legend_title="Engine ID",
        height=700,
        width=1000,
    )
    display(fig)

plot_engine_issues_plotly(df=df_engines, tc="single")
plot_engine_issues_plotly(df=df_engines, tc="twin")

In [0]:
import plotly.graph_objs as go
from scipy.stats import ttest_ind

def plot_var_vs_number_tc(df, var_cols):
    colors = {True: "red", False: "green"}
    if isinstance(var_cols, str):
        var_cols = [var_cols]
    for var_col in var_cols:
        pdf = df.select(var_col, "breakdown", "number_tc").toPandas()
        fig = go.Figure()

        # Ensure x-axis is categorical and sorted alphabetically
        pdf["number_tc"] = pdf["number_tc"].astype(str)
        tc_categories = sorted(pdf["number_tc"].unique())

        for breakdown_val in [True, False]:
            group = pdf[pdf["breakdown"] == breakdown_val]
            fig.add_trace(
                go.Scatter(
                    x=group["number_tc"],
                    y=group[var_col],
                    mode="markers",
                    name=f"Breakdown: {breakdown_val}",
                    marker=dict(color=colors[breakdown_val]),
                )
            )

        var_breakdown = pdf[pdf["breakdown"] == True][var_col]
        var_no_breakdown = pdf[pdf["breakdown"] == False][var_col]
        stat, pval = ttest_ind(var_breakdown, var_no_breakdown, equal_var=False)

        if pval < 0.05 and not var_breakdown.empty:
            min_sep = var_breakdown.min()
            fig.add_shape(
                type="line",
                x0=tc_categories[0],
                x1=tc_categories[-1],
                y0=min_sep,
                y1=min_sep,
                line=dict(color="blue", dash="dash"),
            )
            fig.add_annotation(
                x=tc_categories[-1],
                y=min_sep,
                text="",
                xref="x",
                yref="y",
                showarrow=False,
                font=dict(color="blue"),
                yanchor="bottom",
                xanchor="center"
            )

        fig.update_layout(
            title=f"Number_TC vs {var_col} by Breakdown",
            xaxis_title="Number of Turbochargers (number_tc)",
            yaxis_title=f"{var_col}",
            legend_title="Breakdown",
            height=600,
            width=450,
            xaxis=dict(type="category", categoryorder="array", categoryarray=tc_categories),
        )
        display(fig)

In [0]:
import plotly.subplots as sp

var_cols = [
    "op_set_1",
    "op_set_2",
    "op_set_3",
    "oph",
    "bmep",
    "ng_imp",
    "rpm_max",
    "number_up"
]

pdf = df.select(var_cols + ["breakdown", "number_tc"]).toPandas()
colors = {True: "red", False: "green"}
tc_categories = sorted(pdf["number_tc"].astype(str).unique())

fig = sp.make_subplots(rows=3, cols=3, subplot_titles=var_cols)

for idx, var_col in enumerate(var_cols):
    row = idx // 3 + 1
    col = idx % 3 + 1
    for breakdown_val in [True, False]:
        group = pdf[pdf["breakdown"] == breakdown_val]
        fig.add_trace(
            go.Scatter(
                x=group["number_tc"].astype(str),
                y=group[var_col],
                mode="markers",
                name=f"{var_col} - Breakdown: {breakdown_val}" if idx == 0 else None,
                marker=dict(color=colors[breakdown_val]),
                showlegend=(idx == 0),
            ),
            row=row,
            col=col,
        )
    from scipy.stats import ttest_ind
    var_breakdown = pdf[pdf["breakdown"] == True][var_col]
    var_no_breakdown = pdf[pdf["breakdown"] == False][var_col]
    stat, pval = ttest_ind(var_breakdown, var_no_breakdown, equal_var=False)
    if pval < 0.05 and not var_breakdown.empty:
        min_sep = var_breakdown.min()
        fig.add_shape(
            type="line",
            x0=tc_categories[0],
            x1=tc_categories[-1],
            y0=min_sep,
            y1=min_sep,
            line=dict(color="blue", dash="dash"),
            row=row,
            col=col,
        )

fig.update_layout(
    height=1200,
    width=1400,
    title_text="Number_TC vs Variables by Breakdown (3x3 Grid)",
    legend_title="Breakdown",
)
display(fig)